> Disclaimer
>
> This notebook is not intended to be used and was created in the process of developping Bob.
> Some or many features may have been modified since the creation of this file.
> Use at your own risk. 
>
> Christian Tremblay

In [1]:
from pathlib import Path

from typing import Any

from bob.core import (
    p223,
    Device,
    System,
    get_datagraph,
    bind_model_namespace,
    dump,
)

from bob.equipments.hvac.damper import ElectricalActuatedDamper
from bob.equipments.hvac.coil import ChilledWaterCoil, HotWaterCoil
from bob.equipments.hvac.fan import Fan
from bob.equipments.hvac.filter import Filter
from bob.equipments.hvac.damper import Window
from bob.equipments.hvac.boiler import HotWaterBoiler, ElectricalHotWaterBoiler
from bob.equipments.hvac.valve import WaterValve
from bob.equipments.lighting.light import Light
from bob.equipments.hvac.airhandlingunit import AirHandlingUnit
from bob.equipments.hvac.vav import VAV
from bob.equipments.hvac.pump import Pump
from bob.sensor.temperature import AirTemperatureSensor
from bob.sensor.flow import AirFlowSensor

from bob.space.physical import Building, Floor, Roof, Office, Room, Bathroom, Corridor
from bob.space.hvac import HVACSpace, HVACZone
from bob.space.light import LightingSpace, LightingZone

from bob.connections.air import *
from bob.connections.water import WaterConnection, WaterInletConnectionPoint, WaterOutletConnectionPoint

_namespace = p223

class AgnosticWaterBoiler(Device):
    node_type = p223.AgnosticBoiler
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

class AgnosticWaterCoil(Device):
    node_type = p223.AgnosticCoil
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

class HotWaterTank(Device):
    node_type = p223.HotWaterTank
    waterInlet: WaterInletConnectionPoint
    waterOutlet: WaterOutletConnectionPoint

DHWBoiler = AgnosticWaterBoiler(label='DHW-BOILER')
htgloop_boiler = AgnosticWaterBoiler(label='HTGLOOP-BOILER')
dhw_hot_water_tank = HotWaterTank(label='DHW-TANK')

htg_hot_water_tank = HotWaterTank(label='HTG-TANK')

city_water_tap = WaterConnection(label='CITY-WATER')
dhw_supply_for_house = WaterConnection(label='HouseFaucets')
kitchen_faucet = WaterValve(label='KITCHENFAUCET')
house_drain = WaterConnection(label='HOUSE-DRAIN')
city_drain = WaterConnection(label='CityDrain')

htg_pump = Pump(label='HowWaterPump')
house_hw_supply = WaterConnection(label='HotWaterSupply')
house_hw_return = WaterConnection(label='HotWaterReturn')
joelsofficeheatingvalve = WaterValve(label='joelsofficehtgvlv')
joelsofficeheatingcoil = AgnosticWaterCoil(label='JoelsOfficeCoil')
fillingValve = WaterValve(label='FILL-VLV', comment='Fill Hot Water Loop with water')

city_water_tap >> DHWBoiler.waterInlet
DHWBoiler.waterOutlet >> dhw_hot_water_tank.waterInlet
dhw_hot_water_tank.waterOutlet >> dhw_supply_for_house >> kitchen_faucet.waterInlet
kitchen_faucet.waterOutlet >> city_drain

htg_hot_water_tank.waterOutlet >> htgloop_boiler.waterInlet
htgloop_boiler.waterOutlet >> htg_pump.waterInlet
htg_pump.waterOutlet >> house_hw_supply >> joelsofficeheatingvalve.waterInlet
joelsofficeheatingvalve.waterOutlet >> joelsofficeheatingcoil.waterInlet
joelsofficeheatingcoil.waterOutlet >> house_hw_return
house_hw_return >> htg_hot_water_tank.waterInlet
city_water_tap >> fillingValve.waterInlet
fillingValve.waterOutlet >> house_hw_return

DHWSystem = System(label='DHW', comment="DHW loop in my house")
DHWSystem > [DHWBoiler, dhw_hot_water_tank, dhw_supply_for_house]




<System DHW at N87ca0ccca5b7423ea4b9f4074927c14a>